# 📘 Week 7: Audio & Speech Basics
Welcome to Week 7 of the **Signal Processing Roadmap**! In this notebook, we transition from general signal processing concepts to the rich domain of **audio and speech processing**.

## 🎯 Learning Objectives:
- Understand the fundamentals of digital audio (Sampling Rate, Amplitude, Time-domain visualization).
- Load, save, and play audio waveforms within a Jupyter Notebook.
- Extract time-domain features: **Short-Time Energy (STE)** and **Zero-Crossing Rate (ZCR)**.
- Estimate the **Pitch / Fundamental Frequency ($F_0$)** of a speech sound using the **Autocorrelation method**.
- **Mini Project**: Build a rule-based **Voice Activity Detector (VAD)** to classify audio frames as Voiced, Unvoiced, or Silence.

## 1. Digital Audio Fundamentals & Waveform Generation
Before diving into speech features, let's understand how continuous sound becomes digital audio:
- **Sampling Rate ($f_s$):** The number of samples recorded per second (usually 16 kHz for speech, 44.1 kHz or 48 kHz for high-quality audio).
- **Bit Depth:** The resolution of each sample (e.g., 16-bit signed integers, or 32-bit floats scaled between -1.0 and 1.0).

Let's begin by generating a synthetic audio clip containing:
1. A pure **440 Hz tone** (musical note A4).
2. A frequency-swept **chirp** (transitioning from 200 Hz to 800 Hz).
3. Some background white noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from IPython.display import Audio

# Parameters
fs = 16000  # 16 kHz sampling rate
duration = 2.0  # 2 seconds total duration
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

# 1. 440 Hz Sine wave for first half
tone = np.sin(2 * np.pi * 440 * t[:len(t)//2])

# 2. Linear Chirp from 200 to 800 Hz for second half
t_half = t[len(t)//2:] - t[len(t)//2]
chirp = np.sin(2 * np.pi * (200 + (800 - 200) * t_half / (2 * duration)) * t_half)

# Combine signals and add a tiny bit of white noise
audio_signal = np.concatenate([tone, chirp])
noise = np.random.normal(0, 0.05, len(audio_signal))
audio_signal = audio_signal + noise

# Normalize audio to fit in range [-1.0, 1.0]
audio_signal = audio_signal / np.max(np.abs(audio_signal))

# Save as local WAV file
wav.write("synthetic_audio.wav", fs, (audio_signal * 32767).astype(np.int16))

# Plot the waveform
plt.figure(figsize=(12, 4))
plt.plot(t, audio_signal, color='#1f77b4', alpha=0.7)
plt.title("Synthetic Waveform (Tone + Chirp + Noise)")
plt.xlabel("Time (seconds)")
plt.ylabel("Normalized Amplitude")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Listen to the synthetic sound
Audio(audio_signal, rate=fs)

## 2. Loading and Visualizing Real-World Audio
Let's write code to load the WAV file we just saved using `scipy.io.wavfile.read` or `librosa.load`. 

> **Note on Librosa:** Librosa is a powerful library for audio analysis. By default, `librosa.load` resamples the audio to 22050 Hz and converts it to floating point format in the range `[-1.0, 1.0]`. If you want to keep the original sample rate, you must pass `sr=None`.

In [ ]:
import librosa

# Load using librosa
y, sr = librosa.load("synthetic_audio.wav", sr=None)
print(f"Audio successfully loaded!")
print(f"Sample rate: {sr} Hz")
print(f"Number of samples: {len(y)}")
print(f"Duration: {len(y)/sr:.2f} seconds")

## 3. Feature Extraction: Zero-Crossing Rate (ZCR)
**Zero-Crossing Rate (ZCR)** measures how many times the audio signal crosses the zero value per second.
$$\text{ZCR} = \frac{1}{2N} \sum_{n=1}^{N-1} |\text{sign}(x[n]) - \text{sign}(x[n-1])|$$

### Why is it useful for speech?
- **Voiced Speech:** Sounds like vowels (e.g., "ah", "ee") are periodic, low-frequency, and have a **low ZCR**.
- **Unvoiced Speech:** Fricative consonants (e.g., "s", "f", "sh") behave like high-frequency noise and have a **high ZCR**.
- **Silence/Noise:** Typically has a low ZCR, but can vary depending on background noise features.

Let's implement a frame-by-frame ZCR extraction.

In [ ]:
def extract_zcr(signal, frame_length, hop_length):
    """
    Extracts Zero Crossing Rate on a frame-by-frame basis.
    """
    num_frames = 1 + (len(signal) - frame_length) // hop_length
    zcr_values = []
    
    for i in range(num_frames):
        start_idx = i * hop_length
        end_idx = start_idx + frame_length
        frame = signal[start_idx:end_idx]
        
        # Calculate ZCR
        sign_diff = np.abs(np.diff(np.sign(frame)))
        zcr = np.sum(sign_diff) / (2.0 * frame_length)
        zcr_values.append(zcr)
        
    return np.array(zcr_values)

# Frame parameters
frame_length = 512  # ~32 ms at 16 kHz
hop_length = 256     # ~16 ms overlap

zcr = extract_zcr(y, frame_length, hop_length)

# We can also verify with librosa's built-in feature
librosa_zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length, center=False)[0]

print(f"Custom ZCR shape: {zcr.shape}")
print(f"Librosa ZCR shape: {librosa_zcr.shape}")
assert np.allclose(zcr, librosa_zcr), "Custom ZCR implementation does not match librosa!"

# Plot the waveform alongside ZCR
time_axes_zcr = np.linspace(0, len(y)/sr, len(zcr))

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t, y, color='#1f77b4', alpha=0.7)
ax[0].set_title("Audio Waveform")
ax[0].set_ylabel("Amplitude")
ax[0].grid(True, linestyle='--')

ax[1].plot(time_axes_zcr, zcr, color='#ff7f0e', linewidth=2)
ax[1].set_title("Zero-Crossing Rate (ZCR) Over Time")
ax[1].set_xlabel("Time (seconds)")
ax[1].set_ylabel("ZCR")
ax[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()

## 4. Feature Extraction: Short-Time Energy (STE)
**Short-Time Energy (STE)** is the sum of squared amplitudes within a frame. It represents the volume or power of the audio signal at a specific time window.
$$\text{STE} = \sum_{n=0}^{N-1} x^2[n]$$

For convenience, we can normalize it by the frame length or use Root-Mean-Square (RMS) energy:
$$\text{RMS} = \sqrt{\frac{1}{N} \sum_{n=0}^{N-1} x^2[n]}$$

Let's implement the Short-Time RMS Energy of our audio signal.

In [ ]:
def extract_rms(signal, frame_length, hop_length):
    """
    Extracts Root-Mean-Square Energy on a frame-by-frame basis.
    """
    num_frames = 1 + (len(signal) - frame_length) // hop_length
    rms_values = []
    
    for i in range(num_frames):
        start_idx = i * hop_length
        end_idx = start_idx + frame_length
        frame = signal[start_idx:end_idx]
        
        rms = np.sqrt(np.mean(frame**2))
        rms_values.append(rms)
        
    return np.array(rms_values)

rms = extract_rms(y, frame_length, hop_length)
librosa_rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length, center=False)[0]
assert np.allclose(rms, librosa_rms), "Custom RMS implementation does not match librosa!"

# Plot waveform and RMS Energy
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t, y, color='#1f77b4', alpha=0.7)
ax[0].set_title("Audio Waveform")
ax[0].set_ylabel("Amplitude")
ax[0].grid(True, linestyle='--')

ax[1].plot(time_axes_zcr, rms, color='#2ca02c', linewidth=2)
ax[1].set_title("Short-Time RMS Energy")
ax[1].set_xlabel("Time (seconds)")
ax[1].set_ylabel("RMS Energy")
ax[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()

## 5. Pitch Estimation via Autocorrelation ($F_0$)
The fundamental frequency ($F_0$) corresponds to the perceived pitch of a voiced speech segment.
One classic method to estimate pitch in the time domain is the **Autocorrelation function (ACF)**.

The autocorrelation of a signal $x[n]$ at a lag $\tau$ is:
$$R(\tau) = \sum_{n} x[n] x[n + \tau]$$

For periodic signals, the autocorrelation function $R(\tau)$ will show local peaks at lags corresponding to multiples of the true period of the signal. By searching for the first major peak (ignoring $\tau=0$), we can find the pitch period $T_0$, and compute $F_0 = \frac{f_s}{T_0}$.

Let's write an autocorrelation function to estimate the fundamental frequency ($F_0$) of the 440 Hz tone in the first half of our audio signal.

In [ ]:
def estimate_pitch_acf(frame, fs, fmin=50, fmax=500):
    """
    Estimates the pitch of an audio frame using the Autocorrelation method.
    fmin, fmax set the limits of human pitch search (usually 50 Hz to 500 Hz).
    """
    # Calculate autocorrelation
    n = len(frame)
    acf = np.correlate(frame, frame, mode='full')
    # Keep only the second half (positive lags)
    acf = acf[n - 1:]
    
    # Convert frequency bounds to autocorrelation lag bounds
    lag_min = int(fs / fmax)
    lag_max = int(fs / fmin)
    
    # Find the peak lag within search bounds
    search_range = acf[lag_min:lag_max]
    if len(search_range) == 0:
        return 0.0
    
    peak_lag = np.argmax(search_range) + lag_min
    
    # Convert peak lag to frequency
    f0 = fs / peak_lag
    return f0, acf, lag_min, lag_max

# Test on a clean frame from the 440 Hz portion (first second)
test_frame = y[int(0.2*fs) : int(0.2*fs + 1024)]
f0_est, acf, l_min, l_max = estimate_pitch_acf(test_frame, fs)

print(f"Estimated Pitch (Tone Section): {f0_est:.2f} Hz (Target: 440 Hz)")

# Plot autocorrelation peak
plt.figure(figsize=(10, 4))
plt.plot(acf[:100], label='Autocorrelation')
plt.axvline(x=fs/f0_est, color='r', linestyle='--', label=f'Peak corresponding to {f0_est:.1f}Hz')
plt.title("Autocorrelation Function for Pitch Estimation")
plt.xlabel("Lag (samples)")
plt.ylabel("Autocorrelation Value")
plt.legend()
plt.grid(True)
plt.show()

## 6. Mini Project: Voice Activity Detector (VAD)
A Voice Activity Detector (VAD) classifies segments of audio into speech vs. silence/noise. VAD is critical in telephony, speech recognition, and audio coding to avoid transmitting silence.

We can build a simple rule-based VAD using our time-domain features:
1. **Silence:** Energy is below a certain threshold ($E_{sil}$). ZCR is low.
2. **Voiced Speech:** Energy is high, and ZCR is low.
3. **Unvoiced Speech:** Energy is moderate-to-low, but ZCR is high.

Let's test this classification on our synthetic audio signal.

In [ ]:
# Set thresholds based on feature plots
energy_threshold = 0.15
zcr_threshold = 0.05

vad_labels = []  # List to hold classification for each frame

for i in range(len(rms)):
    frame_energy = rms[i]
    frame_zcr = zcr[i]
    
    if frame_energy < energy_threshold:
        if frame_zcr > zcr_threshold:
            vad_labels.append(1)  # Unvoiced (high ZCR, low energy)
        else:
            vad_labels.append(0)  # Silence / Background noise
    else:
        if frame_zcr < zcr_threshold:
            vad_labels.append(2)  # Voiced (high energy, low ZCR)
        else:
            vad_labels.append(1)  # Transition/Unvoiced (high energy, high ZCR)

vad_labels = np.array(vad_labels)

# Plot classification
plt.figure(figsize=(12, 5))
plt.plot(t, y, label="Waveform", alpha=0.5)
# Upsample the frame labels for plotting
upsampled_labels = np.repeat(vad_labels, hop_length)
upsampled_t = np.linspace(0, len(y)/sr, len(upsampled_labels))

plt.plot(upsampled_t, upsampled_labels / 2.0, 'r-', label="VAD State (0: Silence, 0.5: Unvoiced, 1: Voiced)", linewidth=2)
plt.title("Voice Activity Detection (VAD) Output")
plt.xlabel("Time (seconds)")
plt.ylabel("Signal / State")
plt.legend()
plt.grid(True)
plt.show()

## ✅ Reflection & Exercises
- **VAD Tuning:** What happens if the background noise increases? Try changing the noise level in Section 1 to `0.3` and see how it affects your VAD results. What thresholds need tuning?
- **Autocorrelation limits:** Why did we set `fmin=50` and `fmax=500` for the pitch estimation? What issues might occur if we search for a pitch peak at lag=1 (very high frequency)?